In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import zstandard as zstd
import lz4.frame
import snappy  # python-snappy
import xgboost as xgb

# =========================
# 0. 경로 / 설정
# =========================
DATA_DIR = Path("data")          # 학습용 원본들
RAW_DIR = Path("raw")            # 벤치마크용 원본들
TRAIN_CSV_PATH = Path("train_features_all_samples.csv")

# 청크 크기 (압축/피처 공통)
CHUNK_SIZES_BYTES = [
    2 * 1024 * 1024,   # 2MB
    4 * 1024 * 1024,   # 4MB
    8 * 1024 * 1024,   # 8MB
    16 * 1024 * 1024,  # 16MB
]

# 샘플링 크기 (피처용만, 압축은 항상 full-chunk)
SAMPLE_SIZES = [
    128 * 1024,        # 128KB
    256 * 1024,        # 256KB
    512 * 1024,        # 512KB
    1024 * 1024,       # 1MB
]
SAMPLE_LABELS = {s: f"{s // 1024}KB" for s in SAMPLE_SIZES}

print("사용할 청크 크기 (bytes):", CHUNK_SIZES_BYTES)
print("사용할 샘플링 크기 (bytes):", SAMPLE_SIZES, "=>", SAMPLE_LABELS)

# =========================
# 1. 코덱 관련 설정
# =========================
CODECS = ["zstd", "lz4", "snappy"]
CODEC_TO_LABEL = {c: i for i, c in enumerate(CODECS)}
LABEL_TO_CODEC = {i: c for c, i in CODEC_TO_LABEL.items()}

print("CODEC_TO_LABEL:", CODEC_TO_LABEL)


# =========================
# 2. cost 함수
# =========================
def compute_normalized_cost(original_size: int, compressed_size: int, seconds: float) -> float:
    """
    cost = (compressed_size / original_size) * (seconds / original_size)
    - 압축률과 시간 둘 다 고려하는 비용
    """
    if original_size == 0:
        return float("inf")
    compression_ratio = compressed_size / original_size
    time_per_byte = seconds / original_size
    return compression_ratio * time_per_byte


# =========================
# 3. 샘플링 함수
# =========================
def sample_bytes(data: bytes, max_len: int) -> bytes:
    """
    큰 청크에서 max_len 정도만 균일 샘플링해서 피처 계산.
    압축 자체는 항상 full chunk 기준으로 진행.
    """
    n = len(data)
    if n <= max_len:
        return data
    stride = n // max_len
    return data[::stride]


# =========================
# 4. 피처 추출 (NumPy-only)
# =========================
FEATURE_KEYS = [
    "entropy",
    "frac_zero",
    "frac_ff",
    "frac_ascii_printable",
    "frac_control",
    "frac_space",
    "frac_newline",
    "run_mean",
    "run_std",
    "runs_per_byte",
]


def compute_features_arr_numpy(arr: np.ndarray) -> np.ndarray:
    """
    arr: uint8 1D 배열
    - np.bincount로 값 분포
    - np.diff로 run-length 추출
    """
    length = arr.size
    out = np.zeros(10, dtype=np.float64)
    if length == 0:
        return out

    length_f = float(length)

    # 1) 0~255 counts
    counts = np.bincount(arr, minlength=256)

    zero = counts[0]
    ff = counts[255]
    ascii_printable = counts[32:127].sum()         # 0x20 ~ 0x7E
    control = counts[:32].sum() - counts[10]       # 0x00~0x1F, 개행 제외
    space = counts[32]
    newline = counts[10]

    frac_zero = zero / length_f
    frac_ff = ff / length_f
    frac_ascii_printable = ascii_printable / length_f
    frac_control = control / length_f
    frac_space = space / length_f
    frac_newline = newline / length_f

    # 2) entropy
    p = counts[counts > 0] / length_f
    ent = -np.sum(p * np.log2(p))

    # 3) run-length (벡터화)
    if length == 1:
        runs = np.array([1], dtype=np.int64)
    else:
        changes = np.nonzero(np.diff(arr) != 0)[0] + 1
        idx = np.concatenate(([0], changes, [length]))
        runs = np.diff(idx)

    num_runs = runs.size
    run_mean = runs.mean()
    run_std = runs.std()
    runs_per_byte = num_runs / length_f

    out[:] = [
        ent,
        frac_zero,
        frac_ff,
        frac_ascii_printable,
        frac_control,
        frac_space,
        frac_newline,
        run_mean,
        run_std,
        runs_per_byte,
    ]
    return out


def extract_all_features(data: bytes) -> dict:
    """
    bytes -> uint8 -> NumPy 기반 피처 계산 → dict(FEATURE_KEYS)
    """
    if len(data) == 0:
        return {k: 0.0 for k in FEATURE_KEYS}
    arr = np.frombuffer(data, dtype=np.uint8)
    vals = compute_features_arr_numpy(arr)
    return {key: float(vals[i]) for i, key in enumerate(FEATURE_KEYS)}


# =========================
# 5. FEATURE_COLUMNS
# =========================
# sampling_size_bytes는 메타 정보이므로 피처에서는 제외 (subset 고를 때만 사용)
FEATURE_COLUMNS = FEATURE_KEYS + ["chunk_size_bytes"]
print("FEATURE_COLUMNS:", FEATURE_COLUMNS)


사용할 청크 크기 (bytes): [2097152, 4194304, 8388608, 16777216]
사용할 샘플링 크기 (bytes): [131072, 262144, 524288, 1048576] => {131072: '128KB', 262144: '256KB', 524288: '512KB', 1048576: '1024KB'}
CODEC_TO_LABEL: {'zstd': 0, 'lz4': 1, 'snappy': 2}
FEATURE_COLUMNS: ['entropy', 'frac_zero', 'frac_ff', 'frac_ascii_printable', 'frac_control', 'frac_space', 'frac_newline', 'run_mean', 'run_std', 'runs_per_byte', 'chunk_size_bytes']


In [2]:
import time

# =========================
# 6. CSV 있으면 바로 로드
# =========================
if TRAIN_CSV_PATH.exists():
    print(f"[INFO] CSV 발견: {TRAIN_CSV_PATH}, 로딩 중…")
    df_train = pd.read_csv(TRAIN_CSV_PATH)
    print(f"[INFO] 로딩 완료: {len(df_train)} rows")
else:
    print("[INFO] CSV 없음 → df 생성 시작")

    files = [p for p in DATA_DIR.iterdir() if p.is_file()]
    print("대상 파일 수:", len(files))
    if not files:
        raise RuntimeError("data/ 안에 파일 없음")

    rows = []
    zstd_compressor = zstd.ZstdCompressor()

    total_bytes = 0
    total_chunks = 0

    t_start = time.perf_counter()

    for file_path in files:
        file_size = file_path.stat().st_size
        print(f"\n파일: {file_path.name} ({file_size / (1024*1024):.2f} MB)")

        size_idx = 0
        with file_path.open("rb") as f:
            while True:
                chunk_size = CHUNK_SIZES_BYTES[size_idx]
                size_idx = (size_idx + 1) % len(CHUNK_SIZES_BYTES)

                chunk = f.read(chunk_size)
                if not chunk:
                    break

                original_size = len(chunk)
                if original_size == 0:
                    break

                total_bytes += original_size
                total_chunks += 1

                # ---- (1) full-chunk 기준 코덱별 cost 계산 ----
                codec_costs = {}
                codec_sizes = {}
                codec_times = {}

                # zstd
                t0 = time.perf_counter()
                c1 = zstd_compressor.compress(chunk)
                t1 = time.perf_counter()
                codec_times["zstd"] = t1 - t0
                codec_sizes["zstd"] = len(c1)
                codec_costs["zstd"] = compute_normalized_cost(original_size, codec_sizes["zstd"], codec_times["zstd"])

                # lz4
                t0 = time.perf_counter()
                c2 = lz4.frame.compress(chunk)
                t1 = time.perf_counter()
                codec_times["lz4"] = t1 - t0
                codec_sizes["lz4"] = len(c2)
                codec_costs["lz4"] = compute_normalized_cost(original_size, codec_sizes["lz4"], codec_times["lz4"])

                # snappy
                t0 = time.perf_counter()
                c3 = snappy.compress(chunk)
                t1 = time.perf_counter()
                codec_times["snappy"] = t1 - t0
                codec_sizes["snappy"] = len(c3)
                codec_costs["snappy"] = compute_normalized_cost(original_size, codec_sizes["snappy"], codec_times["snappy"])

                best_codec = min(codec_costs, key=codec_costs.get)
                best_label = CODEC_TO_LABEL[best_codec]

                # ---- (2) 샘플링 크기별로 피처 뽑고 row 생성 ----
                for samp in SAMPLE_SIZES:
                    sampled = sample_bytes(chunk, max_len=samp)
                    feats = extract_all_features(sampled)

                    row = {k: feats[k] for k in FEATURE_KEYS}
                    row["chunk_size_bytes"] = float(original_size)
                    row["sampling_size_bytes"] = float(samp)
                    row["label"] = best_label
                    row["file_name"] = file_path.name

                    rows.append(row)

    t_end = time.perf_counter()
    elapsed = t_end - t_start

    df_train = pd.DataFrame(rows)

    print(f"\n[INFO] df 생성 완료: {len(df_train)} rows")
    print(f"[INFO] 총 청크 수: {total_chunks}")
    print(f"[INFO] 총 처리 바이트: {total_bytes} bytes ({total_bytes / (1024*1024*1024):.2f} GB)")
    print(f"[INFO] 총 소요 시간: {elapsed:.2f} 초")
    if elapsed > 0:
        print(f"[INFO] 처리 속도: {total_bytes / (1024*1024*elapsed):.2f} MB/s")

    print("\n[INFO] CSV 저장 중…")
    df_train.to_csv(TRAIN_CSV_PATH, index=False)
    print(f"[INFO] 저장 완료 → {TRAIN_CSV_PATH}")


[INFO] CSV 없음 → df 생성 시작
대상 파일 수: 4

파일: 2024-01-01-0.json (479.94 MB)

파일: 2024-01-01-2.json (436.63 MB)

파일: chromium-130.0.6723.70.tar (23432.34 MB)

파일: wikipedia.xml (69892.02 MB)

[INFO] df 생성 완료: 50280 rows
[INFO] 총 청크 수: 12570
[INFO] 총 처리 바이트: 98818780168 bytes (92.03 GB)
[INFO] 총 소요 시간: 1247.68 초
[INFO] 처리 속도: 75.53 MB/s

[INFO] CSV 저장 중…
[INFO] 저장 완료 → train_features_all_samples.csv


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def bucket_chunk_size(size_bytes: float) -> int:
    """
    청크 크기를 가장 가까운 2/4/8/16 MB 버킷으로 매핑
    """
    best = min(CHUNK_SIZES_BYTES, key=lambda cs: abs(size_bytes - cs))
    return int(best / (1024 * 1024))


all_results = {}

for samp in SAMPLE_SIZES:
    print("\n" + "=" * 80)
    print(f"=== 샘플링 크기 {SAMPLE_LABELS[samp]} ({samp} bytes) 기준 모델 학습 ===")

    df_s = df_train[df_train["sampling_size_bytes"] == float(samp)].copy()
    if df_s.empty:
        print("  → 해당 샘플링 크기에 대한 데이터가 없습니다. 스킵.")
        continue

    X = df_s[FEATURE_COLUMNS].astype(np.float32).values
    y = df_s["label"].astype(int).values

    print("  X shape:", X.shape, " y shape:", y.shape)
    label_counts = pd.Series(y).value_counts().sort_index()
    print("  라벨 분포:")
    print(" ", label_counts.rename(index=LABEL_TO_CODEC).to_dict())

    # ----- 클래스 가중치 (inverse frequency) -----
    num_classes = len(CODECS)
    total_samples = len(y)

    class_weight = {}
    for label, cnt in label_counts.items():
        class_weight[label] = total_samples / (num_classes * cnt)

    print("  클래스 가중치:")
    for lab, w in class_weight.items():
        print(f"    {LABEL_TO_CODEC[lab]}: {w:.3f}")

    sample_weight = np.array([class_weight[lab] for lab in y], dtype=np.float32)

    # ----- train / valid split -----
    X_train, X_valid, y_train, y_valid, w_train, w_valid = train_test_split(
        X, y, sample_weight,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, feature_names=FEATURE_COLUMNS)
    dvalid = xgb.DMatrix(X_valid, label=y_valid, weight=w_valid, feature_names=FEATURE_COLUMNS)

    # ----- XGBoost 파라미터 -----
    params = {
        "objective": "multi:softprob",
        "num_class": len(CODECS),
        "max_depth": 6,
        "eta": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "tree_method": "hist",
        "eval_metric": "mlogloss",
    }

    num_boost_round = 200

    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        verbose_eval=20,
    )

    # ----- 검증 성능 -----
    pred_prob = bst.predict(dvalid)
    y_pred = pred_prob.argmax(axis=1)

    print("\n  === Validation Report ===")
    print(classification_report(
        y_valid,
        y_pred,
        target_names=CODECS,
    ))

    # ----- breakdown (코덱별, 청크사이즈별) -----
    valid_df = pd.DataFrame(X_valid, columns=FEATURE_COLUMNS)
    valid_df["true_label"] = y_valid
    valid_df["pred_label"] = y_pred
    valid_df["true_codec"] = valid_df["true_label"].map(LABEL_TO_CODEC)
    valid_df["pred_codec"] = valid_df["pred_label"].map(LABEL_TO_CODEC)
    valid_df["correct"] = (valid_df["true_label"] == valid_df["pred_label"]).astype(int)
    valid_df["chunk_size_mb"] = valid_df["chunk_size_bytes"].apply(bucket_chunk_size)

    overall_acc = valid_df["correct"].mean()
    codec_acc = valid_df.groupby("true_codec")["correct"].mean()
    size_acc = valid_df.groupby("chunk_size_mb")["correct"].mean()
    size_codec_acc = (
        valid_df.groupby(["chunk_size_mb", "true_codec"])["correct"]
        .mean()
        .unstack()
        .reindex(columns=CODECS)
    )

    print(f"\n  === 샘플 {SAMPLE_LABELS[samp]} 전체 정확도 ===")
    print(f"    Accuracy: {overall_acc:.4f}")

    print("\n  === 코덱별 정확도 ===")
    print(codec_acc)

    print("\n  === 청크사이즈별 정확도 ===")
    print(size_acc)

    print("\n  === 청크사이즈 × 코덱별 정확도 ===")
    print(size_codec_acc)

    # 결과 저장 (나중에 비교용)
    all_results[samp] = {
        "overall_acc": overall_acc,
        "codec_acc": codec_acc,
        "size_acc": size_acc,
        "size_codec_acc": size_codec_acc,
    }

    # ----- 모델 저장 -----
    model_path = Path(f"model_sample_{samp // 1024}KB.json")
    bst.save_model(str(model_path))
    print(f"\n  [저장 완료] 모델: {model_path}")

print("\n\n=== 샘플링 크기별 전체 정확도 요약 ===")
for samp, res in all_results.items():
    print(f"  {SAMPLE_LABELS[samp]}: acc = {res['overall_acc']:.4f}")



=== 샘플링 크기 128KB (131072 bytes) 기준 모델 학습 ===
  X shape: (12570, 11)  y shape: (12570,)
  라벨 분포:
  {'zstd': 1777, 'lz4': 8831, 'snappy': 1962}
  클래스 가중치:
    zstd: 2.358
    lz4: 0.474
    snappy: 2.136
[0]	train-mlogloss:1.06706	valid-mlogloss:1.07042
[20]	train-mlogloss:0.73222	valid-mlogloss:0.78822
[40]	train-mlogloss:0.61580	valid-mlogloss:0.70649
[60]	train-mlogloss:0.55760	valid-mlogloss:0.67785
[80]	train-mlogloss:0.51986	valid-mlogloss:0.66734
[100]	train-mlogloss:0.49326	valid-mlogloss:0.66545
[120]	train-mlogloss:0.47267	valid-mlogloss:0.66580
[140]	train-mlogloss:0.45155	valid-mlogloss:0.66749
[160]	train-mlogloss:0.43307	valid-mlogloss:0.66887
[180]	train-mlogloss:0.41495	valid-mlogloss:0.67272
[199]	train-mlogloss:0.39787	valid-mlogloss:0.67557

  === Validation Report ===
              precision    recall  f1-score   support

        zstd       0.68      0.79      0.73       356
         lz4       0.87      0.90      0.89      1766
      snappy       0.54      0.37      

In [9]:
import time

# =========================
# 1. 벤치마크 설정
# =========================

# 🔧 (1) 벤치마크에 사용할 샘플링 크기 (반드시 이미 학습/저장한 것 중에서 하나)
BENCH_SAMPLE_SIZE = 128 * 1024  # 128*1024, 256*1024, 512*1024, 1024*1024 중 선택
BENCH_SAMPLE_LABEL = SAMPLE_LABELS[BENCH_SAMPLE_SIZE]

# 🔧 (2) 벤치마크에 사용할 청크 사이즈(MB 단위 리스트)
BENCH_CHUNK_SIZES_MB = [16, 16]  # 예: [4], [4, 8], [8, 16] 등으로 바꿔가며 실험
BENCH_CHUNK_SIZES_BYTES = [mb * 1024 * 1024 for mb in BENCH_CHUNK_SIZES_MB]

print(f"[INFO] 벤치마크용 샘플링 크기: {BENCH_SAMPLE_LABEL} ({BENCH_SAMPLE_SIZE} bytes)")
print(f"[INFO] 벤치마크용 청크 크기(MB): {BENCH_CHUNK_SIZES_MB}")
print(f"[INFO] 벤치마크용 청크 크기(bytes): {BENCH_CHUNK_SIZES_BYTES}")

# 🔧 (3) 사용할 모델 경로
MODEL_PATH = Path(f"model_sample_{BENCH_SAMPLE_SIZE // 1024}KB.json")
print(f"[INFO] 사용할 모델: {MODEL_PATH}")

booster = xgb.Booster()
booster.load_model(str(MODEL_PATH))

# =========================
# 2. raw 디렉토리 파일 목록
# =========================
raw_files = [p for p in RAW_DIR.iterdir() if p.is_file()]
print("벤치마크 대상 파일 수:", len(raw_files))
for p in raw_files:
    print(f" - {p.name} ({p.stat().st_size / (1024*1024):.2f} MB)")

if not raw_files:
    raise RuntimeError("raw/ 안에 벤치마크용 파일이 없습니다.")


# =========================
# 3. 벤치마크 루프
# =========================
zstd_compressor = zstd.ZstdCompressor()

# 메소드별 통계: 총 압축 바이트, 시간
methods = ["zstd_only", "lz4_only", "snappy_only", "oracle", "model"]
stats = {}
for m in methods:
    stats[m] = {
        "compressed_bytes": 0,
        "compress_time": 0.0,   # 순수 압축 시간
    }

# model 전용 추가 타임
stats["model"]["feat_time"] = 0.0     # 샘플링 + 피처 추출
stats["model"]["pred_time"] = 0.0     # XGBoost 예측

total_original_bytes = 0
total_chunks = 0

t_bench_start = time.perf_counter()

for file_path in raw_files:
    file_size = file_path.stat().st_size
    print(f"\n[벤치마크] 파일: {file_path.name} ({file_size / (1024*1024):.2f} MB)")

    size_idx = 0
    with file_path.open("rb") as f:
        while True:
            # 🔁 벤치마크용 청크크기 순환
            chunk_size = BENCH_CHUNK_SIZES_BYTES[size_idx]
            size_idx = (size_idx + 1) % len(BENCH_CHUNK_SIZES_BYTES)

            chunk = f.read(chunk_size)
            if not chunk:
                break

            original_size = len(chunk)
            if original_size == 0:
                break

            total_original_bytes += original_size
            total_chunks += 1

            # ---- (1) full chunk 기준 zstd/lz4/snappy 압축 ----
            codec_sizes = {}
            codec_times = {}
            codec_costs = {}

            # zstd
            t0 = time.perf_counter()
            c1 = zstd_compressor.compress(chunk)
            t1 = time.perf_counter()
            codec_times["zstd"] = t1 - t0
            codec_sizes["zstd"] = len(c1)
            codec_costs["zstd"] = compute_normalized_cost(
                original_size, codec_sizes["zstd"], codec_times["zstd"]
            )

            # lz4
            t0 = time.perf_counter()
            c2 = lz4.frame.compress(chunk)
            t1 = time.perf_counter()
            codec_times["lz4"] = t1 - t0
            codec_sizes["lz4"] = len(c2)
            codec_costs["lz4"] = compute_normalized_cost(
                original_size, codec_sizes["lz4"], codec_times["lz4"]
            )

            # snappy
            t0 = time.perf_counter()
            c3 = snappy.compress(chunk)
            t1 = time.perf_counter()
            codec_times["snappy"] = t1 - t0
            codec_sizes["snappy"] = len(c3)
            codec_costs["snappy"] = compute_normalized_cost(
                original_size, codec_sizes["snappy"], codec_times["snappy"]
            )

            # ---- (2) 단일 코덱 baseline ----
            stats["zstd_only"]["compressed_bytes"] += codec_sizes["zstd"]
            stats["zstd_only"]["compress_time"] += codec_times["zstd"]

            stats["lz4_only"]["compressed_bytes"] += codec_sizes["lz4"]
            stats["lz4_only"]["compress_time"] += codec_times["lz4"]

            stats["snappy_only"]["compressed_bytes"] += codec_sizes["snappy"]
            stats["snappy_only"]["compress_time"] += codec_times["snappy"]

            # ---- (3) oracle: cost 최소 코덱 선택 ----
            best_codec = min(codec_costs, key=codec_costs.get)
            stats["oracle"]["compressed_bytes"] += codec_sizes[best_codec]
            stats["oracle"]["compress_time"] += codec_times[best_codec]

            # ---- (4) model-based: 샘플링 + 피처 + 예측 + 압축시간 ----
            #   4-1) 샘플링 + 피처 추출 시간
            t_feat0 = time.perf_counter()
            sampled = sample_bytes(chunk, max_len=BENCH_SAMPLE_SIZE)
            feats = extract_all_features(sampled)
            t_feat1 = time.perf_counter()
            feat_time = t_feat1 - t_feat0

            #   4-2) 모델 예측 시간
            t_pred0 = time.perf_counter()
            feat_vec = np.array(
                [[feats[k] for k in FEATURE_KEYS] + [float(original_size)]],
                dtype=np.float32,
            )
            drow = xgb.DMatrix(feat_vec, feature_names=FEATURE_COLUMNS)
            prob = booster.predict(drow)[0]
            t_pred1 = time.perf_counter()
            pred_time = t_pred1 - t_pred0

            pred_label = int(prob.argmax())
            pred_codec = LABEL_TO_CODEC[pred_label]

            #   4-3) 결과 반영
            stats["model"]["compressed_bytes"] += codec_sizes[pred_codec]
            stats["model"]["compress_time"] += codec_times[pred_codec]
            stats["model"]["feat_time"] += feat_time
            stats["model"]["pred_time"] += pred_time

t_bench_end = time.perf_counter()
elapsed_bench = t_bench_end - t_bench_start

print(f"\n[벤치마크 완료] 총 청크 수: {total_chunks}")
print(
    f"[벤치마크] 총 원본 바이트: {total_original_bytes} bytes "
    f"({total_original_bytes / (1024*1024*1024):.2f} GB)"
)
print(f"[벤치마크] 총 측정 시간(코덱 3개+모델): {elapsed_bench:.2f} 초")

# =========================
# 4. 메소드별 결과 요약 + 표 만들기
# =========================
rows = []

for m in methods:
    comp_bytes = stats[m]["compressed_bytes"]
    compress_time = stats[m]["compress_time"]

    if m == "model":
        feat_time = stats["model"]["feat_time"]
        pred_time = stats["model"]["pred_time"]
        total_time = compress_time + feat_time + pred_time
    else:
        feat_time = 0.0
        pred_time = 0.0
        total_time = compress_time

    # 압축률 (작을수록 좋음)
    if total_original_bytes > 0:
        compression_ratio = comp_bytes / total_original_bytes
    else:
        compression_ratio = float("inf")

    # 처리 속도: total_time 기준
    if total_time > 0:
        throughput_mb_s = (total_original_bytes / (1024 * 1024)) / total_time
    else:
        throughput_mb_s = float("inf")

    rows.append(
        {
            "method": m,
            "compressed_bytes": comp_bytes,
            "compress_time_sec": compress_time,
            "feat_time_sec": feat_time,
            "pred_time_sec": pred_time,
            "total_time_sec": total_time,
            "compression_ratio": compression_ratio,
            "throughput_MB_per_s": throughput_mb_s,
        }
    )

df_bench = pd.DataFrame(rows).set_index("method")

# oracle 기준 상대 시간/압축률
oracle_total_time = df_bench.loc["oracle", "total_time_sec"]
oracle_ratio = df_bench.loc["oracle", "compression_ratio"]

df_bench["rel_time_vs_oracle"] = df_bench["total_time_sec"] / oracle_total_time
df_bench["rel_ratio_vs_oracle"] = df_bench["compression_ratio"] / oracle_ratio

print("\n=== 메소드별 압축 결과 요약 (raw 디렉토리 기준) ===")
display(df_bench)


[INFO] 벤치마크용 샘플링 크기: 128KB (131072 bytes)
[INFO] 벤치마크용 청크 크기(MB): [16, 16]
[INFO] 벤치마크용 청크 크기(bytes): [16777216, 16777216]
[INFO] 사용할 모델: model_sample_128KB.json
벤치마크 대상 파일 수: 5
 - 2024-01-01-1.json (598.21 MB)
 - chromium-122.0.6261.57.tar (17392.71 MB)
 - commoncrawl_00065.wet (210.60 MB)
 - sample (17.08 MB)
 - sample1 (95.37 MB)

[벤치마크] 파일: 2024-01-01-1.json (598.21 MB)

[벤치마크] 파일: chromium-122.0.6261.57.tar (17392.71 MB)

[벤치마크] 파일: commoncrawl_00065.wet (210.60 MB)

[벤치마크] 파일: sample (17.08 MB)

[벤치마크] 파일: sample1 (95.37 MB)

[벤치마크 완료] 총 청크 수: 1148
[벤치마크] 총 원본 바이트: 19203591603 bytes (17.88 GB)
[벤치마크] 총 측정 시간(코덱 3개+모델): 160.69 초

=== 메소드별 압축 결과 요약 (raw 디렉토리 기준) ===


,compressed_bytes,compress_time_sec,feat_time_sec,pred_time_sec,total_time_sec,compression_ratio,throughput_MB_per_s,rel_time_vs_oracle,rel_ratio_vs_oracle
method,,,,,,,,,
zstd_only,4517739528,67.962417,0.000000,0.000000,67.962417,0.235255,269.472053,2.209393,0.755882
lz4_only,6298165497,30.157713,0.000000,0.000000,30.157713,0.327968,607.273248,0.980398,1.053773
snappy_only,6630347011,26.429837,0.000000,0.000000,26.429837,0.345266,692.927932,0.859209,1.109352
oracle,5976775055,30.760678,0.000000,0.000000,30.760678,0.311232,595.369579,1.000000,1.000000
model,5446360751,42.322903,2.931285,1.261915,46.516103,0.283612,393.712522,1.512194,0.911254


In [10]:
16기가 청크 결과
256kb 총 48초
128kb 총 46초

SyntaxError: invalid decimal literal (1879870523.py, line 1)